In [ ]:

import torch
import pandas as pd
import joblib
import os
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [ ]:
# Load dataset
df = pd.read_csv("./dataset.csv")
df = df.reset_index()

LABELS = len(df['label_tec'].value_counts())
encoder = LabelEncoder()
encoder.fit(df['label_tec'])
df['enc_label'] = encoder.transform(df['label_tec'])

In [ ]:
MAX_LEN = 512
TRAIN_BATCH_SIZE = 16
VALID_BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-5

In [ ]:
# Tokenizer and model config for CyBERTuned
MODEL_NAME = "s2w-ai/CyBERTuned-SecurityLLM"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(MODEL_NAME, output_hidden_states=True)

In [ ]:
class Triage(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, index):
        sentence = str(self.data.sentence[index])
        inputs = self.tokenizer.encode_plus(
            sentence,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_token_type_ids=False
        )
        ids = inputs['input_ids']
        mask = inputs['attention_mask']
        return {
            'ids': torch.tensor(ids, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.long),
            'targets': torch.tensor(self.data.enc_label[index], dtype=torch.long)
        }

    def __len__(self):
        return len(self.data)

In [ ]:
# Split train/test
train_dataset, test_dataset = train_test_split(df, test_size=0.2, stratify=df['enc_label'], random_state=42)
train_dataset = train_dataset.reset_index(drop=True)
test_dataset = test_dataset.reset_index(drop=True)
training_set = Triage(train_dataset, tokenizer, MAX_LEN)
testing_set = Triage(test_dataset, tokenizer, MAX_LEN)

training_loader = DataLoader(training_set, batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=0)
testing_loader = DataLoader(testing_set, batch_size=VALID_BATCH_SIZE, shuffle=False, num_workers=0)

In [ ]:
# CyBERTuned classifier model
class CyberTunedClassifier(torch.nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.model = AutoModel.from_pretrained(model_name, config=config)
        self.dropout = torch.nn.Dropout(0.3)
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(config.hidden_size, config.hidden_size // 2),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(config.hidden_size // 2, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state
        mean_pooled = torch.sum(hidden * attention_mask.unsqueeze(-1), dim=1) / attention_mask.sum(1, keepdim=True)
        return self.classifier(self.dropout(mean_pooled))

        # I had sought AI help to build this function

In [ ]:
# Initialize model
model = CyberTunedClassifier(MODEL_NAME, LABELS)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
# Training loop
def train(epoch):
    model.train()
    total_loss = 0
    for i, data in enumerate(training_loader):
        ids = data['ids'].to(device)
        mask = data['mask'].to(device)
        targets = data['targets'].to(device)

        optimizer.zero_grad()
        outputs = model(ids, mask)
        loss = loss_function(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch + 1}, Loss: {total_loss / len(training_loader):.4f}")

# Train all epochs
for epoch in range(EPOCHS):
    train(epoch)

In [ ]:
# Accuracy check
def check_accuracy(loader, model, device='cuda' if torch.cuda.is_available() else 'cpu', return_preds=False):
    num_correct = 0
    num_samples = 0
    model.eval()

    all_sentences = []
    all_predictions = []
    all_targets = []
    all_logits = []

    with torch.no_grad():
        for i, data in enumerate(loader, 0):
            ids = data['ids'].to(device)
            mask = data['mask'].to(device)
            targets = data['targets'].to(device)

            outputs = model(ids, mask)
            _, preds = torch.max(outputs, dim=1)

            num_correct += (preds == targets).sum().item()
            num_samples += targets.size(0)

            all_predictions.extend(preds.cpu().tolist())
            all_targets.extend(targets.cpu().tolist())
            all_logits.append(outputs.cpu())
            if 'sentence' in data:
                all_sentences.extend(data['sentence'])

    acc = 100.0 * num_correct / num_samples
    print(f"Got {num_correct} / {num_samples} correct with accuracy {acc:.2f}%")

    if return_preds:
        return {
            "accuracy": acc,
            "predictions": all_predictions,
            "targets": all_targets,
            "logits": torch.cat(all_logits),
            "sentences": all_sentences if all_sentences else None
        }

check_accuracy(testing_loader, model)

In [ ]:
# Save artifacts
os.makedirs("mitre_model_cybertuned", exist_ok=True)
torch.save(model.state_dict(), "mitre_model_cybertuned/cybertuned_model.pt")
tokenizer.save_pretrained("mitre_model_cybertuned")
joblib.dump(encoder, "mitre_model_cybertuned/label_encoder.pkl")

# Zip for download
import shutil
from google.colab import files

shutil.make_archive("mitre_model_cybertuned", 'zip', "mitre_model_cybertuned")
files.download("mitre_model_cybertuned.zip")

